#### Import Multi-layer Signalling Model (MSM)

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / 'src'))
from msm_model import *


#### Single simulation

In [ ]:
# Notch-Delta parameters
params = dict(k=2, h=8, Ka=10**-1, Kr=10**-3, nu=1, t_final=1000, hs=1.9, sim_number=1, omega_type='exp')
parser=argparse.ArgumentParser()
[parser.add_argument(f"--{n}",type=type(v),default=v,
                     choices=["exp","cnt","lin","exp0"] if n=="omega_type" else None)
 for n,v in params.items()]
args,_=parser.parse_known_args()
k,h,Ka,Kr,nu,t_final,hs,sim_number,omega_type=(
    args.k,args.h,args.Ka,args.Kr,args.nu,args.t_final,args.hs,args.sim_number,args.omega_type)
h1, h2, h3, h4, h5, h6 = [height_set(diam_apical_dict[wingr], hs=hs) for wingr in wing_regions]
omega_func=omega_map[omega_type]
tag = f"{[k,h,Ka,Kr,nu,t_final,hs,omega_type]}"

# Heights
height_list = [h1, h2, h3, h4, h5, h6]
heights_dict = dict(zip(wing_regions, height_list))

# Wing disc
wing_region = 'wd_2_mbs' # Wing disc selection
Lmax = 300 # Signalling depth in μm

result = compute_band_distance(
    wing_region,
    omega_func=omega_func,
    Lmax=Lmax,
    sim_number=1,
    quad_method='simpson',
    height=heights_dict[wing_region],
    plotQ=True, graphsaveQ=False,
    normalQ=False,
    alpha=0,
    degen_T=1.,
    y_shift_steps=20,
    t_final=t_final,
    epsmodelQ=False, eps=0., prot_len=0.,
    k=k, h=h, Ka=Ka, Kr=Kr, nu=nu, dt=dt,
    randomQ=True
)
print(f"SOP spacing: {result[0][0.1]}\nDegenerate pattern: {result[2]}")


#### Spacing plots

In [ ]:
# Notch-Delta parameters
params = dict(k=8, h=8, Ka=10**-2, Kr=10**-4.5, nu=1., t_final=1000, hs=1.85, sim_number=20, omega_type='exp')
parser=argparse.ArgumentParser()
[parser.add_argument(f"--{n}",type=type(v),default=v,
                     choices=["exp","cnt","lin","exp0"] if n=="omega_type" else None)
 for n,v in params.items()]
args,_=parser.parse_known_args()
k,h,Ka,Kr,nu,t_final,hs,sim_number,omega_type=(
    args.k,args.h,args.Ka,args.Kr,args.nu,args.t_final,args.hs,args.sim_number,args.omega_type)
h1, h2, h3, h4, h5, h6 = [height_set(diam_apical_dict[wingr], hs=hs) for wingr in wing_regions]
omega_func=omega_map[omega_type]
tag = f"{[k,h,Ka,Kr,nu,t_final,hs,omega_type]}"

# Heights
height_list = [h1, h2, h3, h4, h5, h6]
heights_dict = dict(zip(wing_regions, height_list))

# Wing discs
wing_regionsl = ['wd_1', 'wd_2', 'wd_3']

# Other settings
threshold = 0.1
degen_T = 1.
normalQ = False
spsteps = 15

# ETA setup
t0 = time.time()
total_jobs = spsteps * 3 * 2
job_count = 0

# Spacing vs 3D contacts
Lmax_list = np.linspace(0.5, 25, spsteps)
spacing_dict_exp = {region: [] for region in wing_regionsl}
it = 0
for Lmax in Lmax_list:
    for region in wing_regionsl:
        d, vr, degenQ, avg_bimodal, _, _ = compute_band_distance(
            region,
            omega_func=omega_func,
            Lmax=Lmax,
            sim_number=sim_number,
            quad_method='simpson',
            height=heights_dict[region],
            degen_T=degen_T,
            normalQ=normalQ,
            t_final=t_final,
            y_shift_steps=20,
            k=k, h=h, Ka=Ka, Kr=Kr, nu=nu, dt=dt
        )
        d = d[threshold]; vr = vr[threshold]
        spacing_dict_exp[region].append([d, vr, degenQ])
        it += 1
        print(f"{it}/{len(Lmax_list)*len(wing_regionsl)}", end="\r")
        # ETA
        job_count += 1
        elapsed = time.time() - t0
        eta = (elapsed / job_count) * (total_jobs - job_count)
        print(f"Progress {job_count}/{total_jobs} | ETA {time.strftime('%H:%M:%S', time.gmtime(eta))}", end='\r')
fancy_plot(spacing_dict_exp, Lmax_list, 'exp', wing_regionsl, degenplotQ=True, ylim=(1.,2.25), errorbarQ=False,
           saveQ=True, title=f'sop_spacing_3D_{tag}', meancolor=shade("#2ca02c", 1), mergeQ=True)

'''
# Spacing vs straightening
alpha_list = np.linspace(0., 1., spsteps)
spacing_dict_exp_straight = {region: [] for region in wing_regionsl}
it = 0
for alpha in alpha_list:
    for region in wing_regionsl:
        d, vr, degenQ, avg_bimodal, _, _ = compute_band_distance(
            region,
            omega_func=omega_exp,
            Lmax=25,
            alpha=alpha,
            sim_number=sim_number,
            quad_method='simpson',
            height=heights_dict[region],
            degen_T=degen_T,
            normalQ=normalQ,
            t_final=t_final,
            y_shift_steps=20,
            str_type='centroid',
            k=k, h=h, Ka=Ka, Kr=Kr, nu=nu, dt=dt
        )
        d = d[threshold]; vr = vr[threshold]
        spacing_dict_exp_straight[region].append([d, vr, degenQ])
        it += 1
        print(f"{it}/{len(alpha_list)*len(wing_regionsl)}", end="\r")
        # ETA
        job_count += 1
        elapsed = time.time() - t0
        eta = (elapsed / job_count) * (total_jobs - job_count)
        print(f"Progress {job_count}/{total_jobs} | ETA {time.strftime('%H:%M:%S', time.gmtime(eta))}", end='\r')
fancy_plot(spacing_dict_exp_straight, alpha_list, 'exp', wing_regionsl, degenplotQ=True, xlim=(0,1), ylim=(1.,2.25), errorbarQ=False,
           x_title='Straightening effect (α)', legend_loc='lower left',
           saveQ=True, title=f'sop_spacing_straight_{tag}', meancolor=shade("#1f77b4", 1), mergeQ=True)
'''

In [ ]:
fancy_plot(spacing_dict_exp, Lmax_list, 'exp', wing_regionsl, degenplotQ=True, ylim=(1.25,2.4), errorbarQ=False,
           saveQ=True, title=f'sop_spacing_3D_{tag}', meancolor=shade("#2ca02c", 1), mergeQ=True)

In [ ]:
# number of neighbours vs straightening percentage
alphas = np.linspace(0,1,15)
A_str_dict = {
    α: straight_adjacency(A_dict, centroids_dict, α)
    for α in alphas
}
deg_wd_1 = plot_straightening_nonapical(A_str_dict, 'wd_1', alphas, saveQ=False)
deg_wd_2 = plot_straightening_nonapical(A_str_dict, 'wd_2', alphas, saveQ=False)
deg_wd_3 = plot_straightening_nonapical(A_str_dict, 'wd_3', alphas, saveQ=False)


In [ ]:
# comparison between mean neighbour change
deg_mean_wd_1 = [np.mean(i) for i in deg_wd_1]
deg_mean_wd_2 = [np.mean(i) for i in deg_wd_2]
deg_mean_wd_3 = [np.mean(i) for i in deg_wd_3]
deg_mean = np.array([
    deg_mean_wd_1,
    deg_mean_wd_2,
    deg_mean_wd_3
], float)
x = np.linspace(0, 100, 15)
line_color = "#D98C00"     # main curve colour
shade_color = "#D98C00"    # shading colour (can be different)
shade_alpha = 0.2          # shading transparency
mean_lw = 4                # line width
ymean = deg_mean.mean(axis=0)
ysd   = deg_mean.std(axis=0)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(x, ymean, color=line_color, linewidth=mean_lw, label="Mean")
ax.fill_between(
    x,
    ymean - ysd,
    ymean + ysd,
    color=shade_color,
    alpha=shade_alpha,
    linewidth=0
)
ax.set_xlabel("Straightening percentage (%)", fontsize=22)
ax.set_ylabel("Median non-apical\nneighbours", fontsize=22)
ax.tick_params(labelsize=18)
ax.set_xlim(0, 100)
ax.grid(True, linestyle="--", alpha=0.6)
leg = ax.legend(fontsize=12, frameon=False, loc="best")
leg.get_frame().set_facecolor("white")
leg.get_frame().set_alpha(0.8)
leg.get_frame().set_edgecolor("black")
leg.set_zorder(10)
ax.set_position([0.10, 0.15, 0.85, 0.75])
plt.savefig("figures/straight_neighbour_comparison.pdf",
            bbox_inches="tight", transparent=True)
plt.show()

#### Sensitivity analysis

In [ ]:
# Notch-Delta parameters
params = dict(k=2, h=8, Ka=10**-1, Kr=10**-3., nu=1., t_final=1000, hs=1.9, sim_number=5, omega_type='exp')
parser=argparse.ArgumentParser()
[parser.add_argument(f"--{n}",type=type(v),default=v,
                     choices=["exp","cnt","lin","exp0"] if n=="omega_type" else None)
 for n,v in params.items()]
args,_=parser.parse_known_args()
k,h,Ka,Kr,nu,t_final,hs,sim_number,omega_type=(
    args.k,args.h,args.Ka,args.Kr,args.nu,args.t_final,args.hs,args.sim_number,args.omega_type)
h1, h2, h3, h4, h5, h6 = [height_set(diam_apical_dict[wingr], hs=hs) for wingr in wing_regions]
omega_func=omega_map[omega_type]
tag = f"{[k,h,Ka,Kr,nu,t_final,hs,omega_type]}"

# Heights
height_list = [h1, h2, h3, h4, h5, h6]
heights_dict = dict(zip(wing_regions, height_list))

# Wing discs
wing_regionsl = ['wd_1', 'wd_2', 'wd_3']

# Other settings
threshold = 0.1
degen_T = 1.
normalQ = False
spsteps = 2

# Sensitivity ranges
Ka_range = [10**i for i in np.linspace(-1.5, -0.5, 9)]
Kr_range = [10**i for i in np.linspace(-5, -1, 9)]

# Dictionary tags
key_params = ["h", "k", "nu", "hs"]
ctx0 = dict(h=h, k=k, nu=nu, hs=hs)
tag = "_".join(f"{p}{ctx0[p]}" for p in key_params)
os.makedirs("data/sensitivity_analysis", exist_ok=True)

def append_row(path, cols, vals):
    new = (not os.path.exists(path)) or (os.path.getsize(path) == 0)
    with open(path, "a") as f:
        if new:
            f.write("# " + "\t".join(cols) + "\n")
        f.write("\t".join(f"{v:.10g}" if isinstance(v, (float, np.floating)) else str(v) for v in vals) + "\n")

# ETA setup
t0 = time.time()
total_jobs = len(Ka_range) * len(Kr_range)
job_count = 0

# Loop
for Ka_i in Ka_range:
    for Kr_i in Kr_range:
        Ka, Kr = Ka_i, Kr_i

        # Spacing vs 3D contacts
        Lmax_list = np.linspace(0.5, 25, spsteps)
        spacing_dict_exp = {region: [] for region in wing_regionsl}
        it = 0
        for Lmax in Lmax_list:
            for region in wing_regionsl:
                d, vr, degenQ, avg_bimodal, _, _ = compute_band_distance(
                    region,
                    omega_func=omega_exp,
                    Lmax=Lmax,
                    sim_number=sim_number,
                    quad_method='simpson',
                    height=heights_dict[region],
                    degen_T=degen_T,
                    normalQ=normalQ,
                    t_final=t_final,
                    y_shift_steps=20,
                    k=k, h=h, Ka=Ka, Kr=Kr, nu=nu, dt=dt
                )
                d = d[threshold]; vr = vr[threshold]
                spacing_dict_exp[region].append([d, vr, degenQ])
                it += 1
                print(f"{it}/{len(Lmax_list)*len(wing_regionsl)}", end="\r")

        # Spacing vs straightening
        alpha_list = np.linspace(0., 1., spsteps)
        spacing_dict_exp_straight = {region: [] for region in wing_regionsl}
        it = 0
        for alpha in alpha_list:
            for region in wing_regionsl:
                d, vr, degenQ, avg_bimodal, _, _ = compute_band_distance(
                    region,
                    omega_func=omega_exp,
                    Lmax=25,
                    alpha=alpha,
                    sim_number=sim_number,
                    quad_method='simpson',
                    height=heights_dict[region],
                    degen_T=degen_T,
                    normalQ=normalQ,
                    t_final=t_final,
                    y_shift_steps=20,
                    str_type='centroid',
                    k=k, h=h, Ka=Ka, Kr=Kr, nu=nu, dt=dt
                )
                d = d[threshold]; vr = vr[threshold]
                spacing_dict_exp_straight[region].append([d, vr, degenQ])
                it += 1
                print(f"{it}/{len(alpha_list)*len(wing_regionsl)}", end="\r")

        # Update and save dictionaries
        dist_3D = np.array([[spacing_dict_exp[w][0][0], spacing_dict_exp[w][1][0]] for w in wing_regionsl])
        dist_straight = np.array([[spacing_dict_exp_straight[w][0][0], spacing_dict_exp_straight[w][1][0]] for w in wing_regionsl])
        
        cols0 = key_params + ["Ka", "Kr"]
        vals0 = [ctx0[p] for p in key_params] + [Ka, Kr]
        append_row(f"data/sensitivity_analysis/spacing_all_{tag}.txt", cols0 + ["dist_3D", "dist_straight"], vals0 + [dist_3D.tolist(), dist_straight.tolist()])


        # ETA
        job_count += 1
        elapsed = time.time() - t0
        eta = (elapsed / job_count) * (total_jobs - job_count)
        print(f"Progress {job_count}/{total_jobs} | ETA {time.strftime('%H:%M:%S', time.gmtime(eta))}")

#### Mbs tests

In [ ]:
# Notch-Delta parameters
params = dict(k=2, h=2, Ka=10**-2.5, Kr=10**-2.5, nu=1., t_final=1000, omf=1, omf2=.5,
              hs=1.55, sim_number=20)
parser = argparse.ArgumentParser()
for name, val in params.items():
    parser.add_argument(f"--{name}", type=type(val), default=val)
args, _ = parser.parse_known_args()
k, h, Ka, Kr, nu, t_final, omf, omf2, hs, sim_number = (args.k, args.h, args.Ka, args.Kr, args.nu, args.t_final, args.omf, args.omf2, args.hs, args.sim_number)
h1, h2, h3, h4, h5, h6 = [height_set(diam_apical_dict[wingr], hs=hs) for wingr in wing_regions]

# Exponential signalling and heights
def omega_exp(z):
    return omf * (np.exp(-0.4 * z) + omf2)
height_list = [h1, h2, h3, h4, h5, h6]
heights_dict = dict(zip(wing_regions, height_list))

# Wing discs
wing_regionsl = ['wd_2_mbs', 'wd_3_mbs', 'wd_8_mbs']

# Other settings
threshold = 0.1
degen_T = 1.
normalQ = False
spsteps = 15

# ETA setup
t0 = time.time()
total_jobs = spsteps * 3 * 2
job_count = 0

# Spacing vs 3D contacts
Lmax_list = np.linspace(0.5, 25, spsteps)
spacing_dict_exp = {region: [] for region in wing_regionsl}
it = 0
for Lmax in Lmax_list:
    for region in wing_regionsl:
        d, vr, degenQ, avg_bimodal, _, _ = compute_band_distance(
            region,
            omega_func=omega_exp,
            Lmax=Lmax,
            sim_number=sim_number,
            quad_method='simpson',
            height=heights_dict[region],
            degen_T=degen_T,
            normalQ=normalQ,
            t_final=t_final,
            y_shift_steps=20,
            k=k, h=h, Ka=Ka, Kr=Kr, nu=nu, dt=dt
        )
        d = d[threshold]; vr = vr[threshold]
        spacing_dict_exp[region].append([d, vr, degenQ])
        it += 1
        print(f"{it}/{len(Lmax_list)*len(wing_regionsl)}", end="\r")
        # ETA
        job_count += 1
        elapsed = time.time() - t0
        eta = (elapsed / job_count) * (total_jobs - job_count)
        print(f"Progress {job_count}/{total_jobs} | ETA {time.strftime('%H:%M:%S', time.gmtime(eta))}", end='\r')
fancy_plot(spacing_dict_exp, Lmax_list, 'exp', wing_regionsl, degenplotQ=True, ylim=(1.,2.5), errorbarQ=False,
           saveQ=True, title=f'sop_spacing_3D_{[k,h,Ka,Kr,nu,t_final,hs]}')

# Spacing vs straightening
alpha_list = np.linspace(0., 1., spsteps)
spacing_dict_exp_straight = {region: [] for region in wing_regionsl}
it = 0
for alpha in alpha_list:
    for region in wing_regionsl:
        d, vr, degenQ, avg_bimodal, _, _ = compute_band_distance(
            region,
            omega_func=omega_exp,
            Lmax=25,
            alpha=alpha,
            sim_number=sim_number,
            quad_method='simpson',
            height=heights_dict[region],
            degen_T=degen_T,
            normalQ=normalQ,
            t_final=t_final,
            y_shift_steps=20,
            str_type='centroid',
            k=k, h=h, Ka=Ka, Kr=Kr, nu=nu, dt=dt
        )
        d = d[threshold]; vr = vr[threshold]
        spacing_dict_exp_straight[region].append([d, vr, degenQ])
        it += 1
        print(f"{it}/{len(alpha_list)*len(wing_regionsl)}", end="\r")
        # ETA
        job_count += 1
        elapsed = time.time() - t0
        eta = (elapsed / job_count) * (total_jobs - job_count)
        print(f"Progress {job_count}/{total_jobs} | ETA {time.strftime('%H:%M:%S', time.gmtime(eta))}", end='\r')
fancy_plot(spacing_dict_exp_straight, alpha_list, 'exp', wing_regionsl, degenplotQ=True, xlim=(0,1), ylim=(1.,2.5), errorbarQ=False,
           x_title='Straightening effect (α)', legend_loc='lower left',
           saveQ=True, title=f'sop_spacing_straight_{[k,h,Ka,Kr,nu,t_final,hs]}')

#### Bimodality analysis

In [ ]:
# Notch-Delta parameters
params = dict(k=2, h=8, Ka=10**-1, Kr=10**-3., nu=1., t_final=50000, hs=1.9, sim_number=1, omega_type='exp')
parser=argparse.ArgumentParser()
[parser.add_argument(f"--{n}",type=type(v),default=v,
                     choices=["exp","cnt","lin","exp0"] if n=="omega_type" else None)
 for n,v in params.items()]
args,_=parser.parse_known_args()
k,h,Ka,Kr,nu,t_final,hs,sim_number,omega_type=(
    args.k,args.h,args.Ka,args.Kr,args.nu,args.t_final,args.hs,args.sim_number,args.omega_type)
h1, h2, h3, h4, h5, h6 = [height_set(diam_apical_dict[wingr], hs=hs) for wingr in wing_regions]
omega_func=omega_map[omega_type]
tag = f"{[k,h,Ka,Kr,nu,t_final,hs,omega_type]}"

# Heights
height_list = [h1, h2, h3, h4, h5, h6]
heights_dict = dict(zip(wing_regions, height_list))

# Wing discs
wing_regionsl = ['wd_1', 'wd_2', 'wd_3']

Lmax = 25 # Signalling depth in μm
delta_sig = []
notch_sig = []

for sim in range(sim_number):
    for wing_region in wing_regionsl:

        for Lmax in [Lmax]:
            result = compute_band_distance(
                wing_region,
                omega_func=omega_func,
                Lmax=Lmax,
                sim_number=1,
                quad_method='simpson',
                height=heights_dict[wing_region],
                plotQ=False, graphsaveQ=False,
                normalQ=False,
                alpha=0,
                degen_T=1.,
                y_shift_steps=20,
                t_final=t_final,
                epsmodelQ=False, eps=0., prot_len=0.,
                k=k, h=h, Ka=Ka, Kr=Kr, nu=nu, dt=dt,
                randomQ=True
            )
            print(f'{sim}/{sim_number}', end='\r')
            #print(f"SOP spacing: {result[0][0.1]}\nDegenerate pattern: {result[2]}")
        
        delta_sig.extend((result[-1])[signalling_labels_dict[wing_region]])
        notch_sig.extend((result[-2])[signalling_labels_dict[wing_region]])


In [ ]:
# Custom figure size
figsize = (6, 4)

plt.figure(figsize=figsize)
plt.hist(delta_sig, bins=30, alpha=0.6, label="Delta")
plt.yscale("log")
plt.xlabel("Delta activity")
plt.ylabel("Frequency")
plt.legend()
plt.title("Distribution of Delta (100 simulations)")
plt.ylim(10**0.5, 10**4)   # adjust to your data range

plt.tight_layout()
plt.savefig("figures/delta_distribution.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# 3. Distributions (histograms)
plt.figure(figsize=figsize)
plt.hist(delta_sig, bins=30, alpha=0.6, label="Delta")
plt.hist(notch_sig, bins=30, alpha=0.6, label="Notch")
plt.yscale("log")
#plt.xscale("log")
plt.xlabel("Activity")
plt.ylabel("Frequency (log scale)")
plt.legend()
plt.title("Distribution of Notch and Delta")
plt.tight_layout()
plt.show()

In [ ]:
# 3. Distributions (histograms)
plt.figure(figsize=figsize)
plt.hist(delta_sig, bins=30, alpha=0.6, label="Delta")
plt.hist(notch_sig, bins=30, alpha=0.6, label="Notch")
plt.yscale("log")
#plt.xscale("log")
plt.xlabel("Activity")
plt.ylabel("Frequency (log scale)")
plt.legend()
plt.title("Distribution of Notch and Delta")
plt.tight_layout()
plt.show()

In [ ]:
# Custom figure size
figsize = (6, 4)

# 1. Plot delta and notch curves
plt.figure(figsize=figsize)
plt.plot(delta_sig, label="Delta")
plt.plot(notch_sig, label="Notch")
plt.xlabel("Cell index")
plt.ylabel("Activity")
plt.legend()
plt.tight_layout()
plt.show()

# 2. Scatter: Notch vs Delta
plt.figure(figsize=figsize)
plt.scatter(notch_sig, delta_sig)
plt.xlabel("Notch")
plt.ylabel("Delta")
plt.title("Notch vs Delta")
plt.tight_layout()
plt.show()

# 3. Distributions (histograms)
plt.figure(figsize=figsize)
plt.hist(delta_sig, bins=30, alpha=0.6, label="Delta")
plt.hist(notch_sig, bins=30, alpha=0.6, label="Notch")
plt.yscale("log")
#plt.xscale("log")
plt.xlabel("Activity")
plt.ylabel("Frequency (log scale)")
plt.legend()
plt.title("Distribution of Notch and Delta")
plt.tight_layout()
plt.show()


#### Supplementary figures

In [ ]:
# Height-distance setup for SOP spacing calculation
regions     = ['wd_1', 'wd_2', 'wd_3']
height_list = [height_set(diam_apical_dict[wingr], hs=1.9) for wingr in regions]
s_shift     = 0.4 # shift in [0,1]
apical_cents = {
    region: np.array(centroids_dict[region][0])[signalling_labels_apical_dict[region]]
    for region in regions
}
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.figure(figsize=(8,6))
ax = plt.gca()
y_span = {}
for color, region, h1 in zip(colors, regions, height_list):
    cents = apical_cents[region]
    x, y = cents[:, 0], cents[:, 1]
    y_min, y_max = y.min(), y.max()
    y_span[region] = y_max-y_min
    y0 = y_min + s_shift * ((y_max - h1) - y_min)
    rect = plt.Rectangle(
        (x.min(), y0),
        np.ptp(x), h1,
        edgecolor=color, facecolor='none', linewidth=2
    )
    ax.add_patch(rect)
    inside = (y >= y0) & (y <= y0 + h1)
    y_in = y[inside]
    std_shift = y_in.std(ddof=1) if y_in.size >= 2 else (0.0 if y_in.size == 1 else np.nan)
    ax.scatter(
        x, y,
        color=color, s=20,
        label=f"WD {wd_dict[region]}: Avg diam: {round(np.mean(diam_apical_dict[region][diam_apical_dict[region] != 0]), 1)}; h={round(h1, 1)}"
    )
ax.set_xlabel('X coordinate (anterior-posterior)')
ax.set_ylabel('Y coordinate (dorso-ventral)')
ax.set_title(f'Apical centroids and distance bands')
ax.legend()
ax.set_aspect('equal', 'box')
plt.tight_layout()
if True:
    plt.savefig('figures/heights_spacing.pdf', bbox_inches='tight', transparent=True)
plt.show()


In [ ]:
# Notch intensity fit
z = 0.5 * np.arange(len(notch_data))
omega = lambda z, A, B, C: A * np.exp(-B * z) + C
p0 = [notch_data[0] - notch_data[-1], 0.1, notch_data[-1]]
(A, B, C), _ = curve_fit(omega, z, notch_data, p0=p0)
gap = 0.5
zs = np.linspace(0, z.max(), 300)
fv = omega(zs, A, B, C)
y0 = np.nanmin(notch_data)
nb = int(z.max() / gap)
plt.figure(figsize=(10, 6))
plt.plot(zs, fv, color="#1f77b4", label="Exponential fit", zorder=2)
for i in range(nb):
    a, b = i * gap, (i + 1) * gap
    m = (zs >= a) & (zs < b)
    plt.fill_between(zs[m], fv[m], y0, color="lightgray", alpha=0.3, zorder=1)
    plt.plot([a, a], [y0, fv[np.argmin(np.abs(zs - a))]], color="black", lw=0.1, alpha=0.5, zorder=1)
xend = nb * gap
plt.plot([xend, xend], [y0, fv[np.argmin(np.abs(zs - xend))]], "k--", lw=0.8, zorder=1)
plt.scatter(z, notch_data, color="#1f77b4", edgecolor="white", s=50, zorder=3, label="Raw data")
eqn = rf"$\omega(z) = {A:.4f}\,e^{{-{B:.4f}\,z}} + {C:.4f}$"
plt.text(2, 0.9 * np.nanmax(fv), eqn, fontsize=12, bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
plt.xlabel("Depth (µm)", fontsize=14)
plt.ylabel("Notch intensity", fontsize=14)
plt.legend(loc="upper right", fontsize=12)
plt.grid(True, ls="--", alpha=0.3)
plt.tight_layout()
if False:
    plt.savefig(f'figures/notch_fit.pdf', bbox_inches='tight', transparent=True)
plt.show()


In [ ]:
# Signalling weight histograms
I = A * (1 - np.exp(-B * 32)) / B + C * 32
A_new = A / I
C_new = C / I
B_new = B
omega_func = lambda z: A_new * np.exp(-B_new * z) + C_new
omega_k_dict = {}
for region in wing_regions:
    wk = compute_omega_k(
        omega_func,
        wing_region=region,
        Lmax=None,
        method='simpson',
        num_simpson=101
    )
    omega_k_dict[region] = wk

blueish = '#80ACD1'
regions = ['wd_1', 'wd_2', 'wd_3']
fig, axes = plt.subplots(len(regions), 1, figsize=(8, 6), sharex=False)

for ax, region in zip(axes, regions):
    wk  = omega_k_dict[region]
    gap = gap_dict[region]
    n   = len(wk)
    edges = np.arange(n) * gap  # left edge of each bar
    ax.bar(edges, wk,
           width=gap,
           align='edge',
           color=blueish,
           edgecolor='white')
    ax.set_xlim(-1, 32.5)
    ax.set_ylim(bottom=np.min(wk)*0.85)
    ax.set_ylabel(r'$\omega_k$')
    ax.grid(True, linestyle='--', alpha=0.3)
    info = f"WD {wd_dict[region]}\n" \
           f"$n$ = {n}\n" \
           r"$\Delta L$" f" = {gap}"
    props = dict(boxstyle="round", facecolor="white", alpha=0.8)
    ax.text(0.98, 0.91,
            info,
            transform=ax.transAxes,
            fontsize=10,
            ha="right",
            va="top",
            bbox=props)
axes[-1].set_xlabel('Depth (µm)')
plt.tight_layout()
if False:
    plt.savefig(f'figures/weights_distribution.pdf', bbox_inches='tight', transparent=True)
plt.show()


#### Stability analysis

In [ ]:
def bimodality_coefficient(delta):
    delta = np.asarray(delta)

    g = skew(delta, bias=False)  # skewness
    k = kurtosis(delta, fisher=False, bias=False)  # Pearson kurtosis

    if k == 0:
        return np.nan  # avoid division by zero

    bc = (g**2 + 1) / k
    return bc

In [ ]:
def plot_bc_heatmap(
    bc_dict,
    vmin=None,
    vmax=None,
    saveQ=False,
    filename="bc_heatmap.pdf"
):
    """
    Plot heatmap of Sarle's BC with:
    - h on y-axis
    - nu on x-axis (log scale)
    - optional colorbar limits
    - optional pdf saving
    """

    # --- Extract sorted unique parameter values ---
    h_vals = sorted({k[0] for k in bc_dict.keys()})
    nu_vals = sorted({k[1] for k in bc_dict.keys()})

    h_vals = np.array(h_vals)
    nu_vals = np.array(nu_vals)

    # --- Build matrix ---
    Z = np.zeros((len(h_vals), len(nu_vals)))

    for i, h in enumerate(h_vals):
        for j, nu in enumerate(nu_vals):
            Z[i, j] = bc_dict.get((h, nu), np.nan)

    # --- Create grid for pcolormesh ---
    H, NU = np.meshgrid(h_vals, nu_vals, indexing='ij')

    fig, ax = plt.subplots(figsize=(8, 5))

    pcm = ax.pcolormesh(
        NU,
        H,
        Z,
        shading='auto',
        vmin=vmin,
        vmax=vmax
    )

    ax.set_xscale('log')

    ax.set_xlabel(r'$\nu$')
    ax.set_ylabel(r'$h$')
    ax.set_title(rf"Sarle's Bimodality Coefficient across (h,$\nu$)")
    ax.invert_yaxis()

    cbar = plt.colorbar(pcm, ax=ax)
    cbar.set_label("BC")

    plt.tight_layout()

    if saveQ:
        plt.savefig(filename, format="pdf")

    plt.show()

In [ ]:
# Notch-Delta parameters
params = dict(k=2, h=8, Ka=10**-1, Kr=10**-3, nu=1, t_final=5000, hs=1.9, sim_number=1, omega_type='exp')
parser=argparse.ArgumentParser()
[parser.add_argument(f"--{n}",type=type(v),default=v,
                     choices=["exp","cnt","lin","exp0"] if n=="omega_type" else None)
 for n,v in params.items()]
args,_=parser.parse_known_args()
k,h,Ka,Kr,nu,t_final,hs,sim_number,omega_type=(
    args.k,args.h,args.Ka,args.Kr,args.nu,args.t_final,args.hs,args.sim_number,args.omega_type)
h1, h2, h3, h4, h5, h6 = [height_set(diam_apical_dict[wingr], hs=hs) for wingr in wing_regions]
omega_func=omega_map[omega_type]
tag = f"{[k,h,Ka,Kr,nu,t_final,hs,omega_type]}"

# Heights
height_list = [h1, h2, h3, h4, h5, h6]
heights_dict = dict(zip(wing_regions, height_list))

# Wing disc
wing_region = 'wd_1' # Wing disc selection
Lmax = 25 # Signalling depth in μm

result = compute_band_distance(
    wing_region,
    omega_func=omega_func,
    Lmax=Lmax,
    sim_number=1,
    quad_method='simpson',
    height=heights_dict[wing_region],
    plotQ=True, graphsaveQ=False,
    normalQ=False,
    alpha=0,
    degen_T=1.,
    y_shift_steps=20,
    t_final=t_final,
    epsmodelQ=False, eps=0., prot_len=0.,
    k=k, h=h, Ka=Ka, Kr=Kr, nu=nu, dt=dt,
    show_labels=False, show_other_layers=False,
    randomQ=True
)
print(f"SOP spacing: {result[0][0.1]}\nDegenerate pattern: {result[2]}")
delta = result[-1]


In [ ]:
h_list = [int(i) for i in np.linspace(2,8,7)]
nu_list = [10**i for i in np.linspace(-2,0,9)]

# Notch-Delta parameters
params = dict(k=2, h=2, Ka=10**-1, Kr=10**-3, nu=0.01, t_final=3000, hs=1.9, sim_number=10, omega_type='exp')
parser=argparse.ArgumentParser()
[parser.add_argument(f"--{n}",type=type(v),default=v,
                     choices=["exp","cnt","lin","exp0"] if n=="omega_type" else None)
 for n,v in params.items()]
args,_=parser.parse_known_args()
k,h,Ka,Kr,nu,t_final,hs,sim_number,omega_type=(
    args.k,args.h,args.Ka,args.Kr,args.nu,args.t_final,args.hs,args.sim_number,args.omega_type)
h1, h2, h3, h4, h5, h6 = [height_set(diam_apical_dict[wingr], hs=hs) for wingr in wing_regions]
omega_func=omega_map[omega_type]
tag = f"{[k,h,Ka,Kr,nu,t_final,hs,omega_type]}"

# Heights
height_list = [h1, h2, h3, h4, h5, h6]
heights_dict = dict(zip(wing_regions, height_list))

# Wing disc
wing_region = 'wd_1' # Wing disc selection
Lmax = 25 # Signalling depth in μm

total_iters = len(h_list) * len(nu_list) * sim_number
iter_count = 0
start_time = time.time()

bc_dict = {}

for h in h_list:
    for nu in nu_list:

        bc_list = []
        for sim in range(sim_number):

            iter_count += 1

            result = compute_band_distance(
                wing_region,
                omega_func=omega_func,
                Lmax=Lmax,
                sim_number=1,
                quad_method='simpson',
                height=heights_dict[wing_region],
                plotQ=False, graphsaveQ=False,
                normalQ=False,
                alpha=0,
                degen_T=1.,
                y_shift_steps=20,
                t_final=t_final,
                epsmodelQ=False, eps=0., prot_len=0.,
                k=k, h=h, Ka=Ka, Kr=Kr, nu=nu, dt=dt,
                randomQ=True
            )

            delta = result[-1]
            bc_list.append(bimodality_coefficient(delta))

            # --- ETA calculation ---
            elapsed = time.time() - start_time
            avg_time = elapsed / iter_count
            remaining = total_iters - iter_count
            eta = avg_time * remaining

            print(
                f"Progress: {iter_count}/{total_iters} | "
                f"h={h}, nu={nu:.3g} | "
                f"ETA: {eta/60:.2f} min",
                end='\r'
            )

        bc_dict[h, nu] = np.mean(bc_list)

print("\nDone.")

In [ ]:
plot_bc_heatmap(bc_dict, vmin=0.98, vmax=max(bc_dict.values()))

#### Extras

In [ ]:
def load_spacing_txt(path, xmin=0., xmax=25.):
    df = pd.read_csv(path, sep=r"\s+|\t", engine="python")

    wing_regions = list(df["wing_region"].unique())

    xs = df.groupby("wing_region")["x"].apply(list)
    xref = np.array(xs.iloc[0], float)

    if not all(np.allclose(v, xref) for v in xs):
        raise ValueError("x grid differs between wing regions in file.")

    n = len(xref)

    Lmax_list = np.linspace(xmin, xmax, n)

    spacing_dict = {}

    for wingr, g in df.groupby("wing_region"):
        g = g.sort_values("x")

        spacing_dict[wingr] = [
            (float(r.d), float(r.vr), bool(int(r.degenQ)))
            for r in g.itertuples(index=False)
        ]

    return spacing_dict, Lmax_list, wing_regions


# usage
tag = "[2, 8, 0.1, 0.001, 1.0, 1000, 1.9, 'exp']"
spacing_dict, Lmax_list, wing_regionsl = load_spacing_txt(f"data/spacing_analysis/sop_spacing_3D_{tag}.txt", xmin=0., xmax=25.)
spacing_dict_straight, Lmax_list_straight, wing_regionsl = load_spacing_txt(f"data/spacing_analysis/sop_spacing_straight_{tag}.txt", xmin=0., xmax=1.)

fancy_plot(spacing_dict, Lmax_list, 'exp', wing_regionsl, degenplotQ=True, ylim=(1.,2.25), errorbarQ=False,
           saveQ=True, title=f'sop_spacing_3D_{tag}', meancolor=shade("#2ca02c", 1), mergeQ=True)
fancy_plot(spacing_dict_straight, Lmax_list_straight, 'exp', wing_regionsl, degenplotQ=True, xlim=(0,1), ylim=(1.,2.25), errorbarQ=False,
           saveQ=True, title=f'sop_spacing_straight_{tag}', meancolor=shade("#1f77b4", 1), mergeQ=True)